In [ ]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

## Parallel CurvesParamapAnalysis

Subclasses `CurvesParamapAnalysis` to override `compute_curves` with a joblib-parallel version.
Each window's curves are computed independently across all CPU cores (`n_jobs=-1`).

In [ ]:
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

from src.time_series_analysis.curves_paramap.framework import CurvesParamapAnalysis


class ParallelCurvesParamapAnalysis(CurvesParamapAnalysis):

    def _compute_window_curves(self, window_ix, window, is_3d):
        """Compute all-frame curves for a single window. Returns (window_ix, curve_dict)."""
        curve_dict = {}
        if is_3d:
            ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
            curve_dict['Window-Axial Start Pix'] = ax_start
            curve_dict['Window-Sagittal Start Pix'] = sag_start
            curve_dict['Window-Coronal Start Pix'] = cor_start
            curve_dict['Window-Axial End Pix'] = ax_end
            curve_dict['Window-Sagittal End Pix'] = sag_end
            curve_dict['Window-Coronal End Pix'] = cor_end
            mask = np.zeros_like(self.seg_data.seg_mask)
            mask[sag_start:sag_end+1, cor_start:cor_end+1, ax_start:ax_end+1] = 1
            n_frames = self.image_data.intensities_for_analysis.shape[3]
            for frame_ix in range(n_frames):
                frame_data = self.image_data.intensities_for_analysis[:, :, :, frame_ix]
                for curve_group in self.curve_groups:
                    curve_names, vals = self.curve_funcs[curve_group](
                        self.image_data, frame_data, mask, **self.analysis_kwargs)
                    for name, val in zip(curve_names, vals):
                        curve_dict.setdefault(name, []).append(val)
        else:
            ax_start, sag_start, ax_end, sag_end = window
            curve_dict['Window-Axial Start Pix'] = ax_start
            curve_dict['Window-Sagittal Start Pix'] = sag_start
            curve_dict['Window-Axial End Pix'] = ax_end
            curve_dict['Window-Sagittal End Pix'] = sag_end
            mask = np.zeros_like(self.seg_data.seg_mask)
            mask[ax_start:ax_end+1, sag_start:sag_end+1] = 1
            n_frames = self.image_data.intensities_for_analysis.shape[0]
            for frame_ix in range(n_frames):
                frame_data = self.image_data.intensities_for_analysis[frame_ix]
                for curve_group in self.curve_groups:
                    curve_names, vals = self.curve_funcs[curve_group](
                        self.image_data, frame_data, mask, **self.analysis_kwargs)
                    for name, val in zip(curve_names, vals):
                        curve_dict.setdefault(name, []).append(val)
        return window_ix, curve_dict

    def compute_curves(self, n_jobs=-1):
        is_3d = self.image_data.intensities_for_analysis.ndim == 4
        if not is_3d and self.image_data.intensities_for_analysis.ndim != 3:
            raise ValueError('Image data must be either 2D+time or 3D+time.')

        results = Parallel(n_jobs=n_jobs)(
            delayed(self._compute_window_curves)(ix, window, is_3d)
            for ix, window in tqdm(enumerate(self.windows), desc='Computing curves', total=len(self.windows))
        )

        self.curves = [curve_dict for _, curve_dict in sorted(results)]

        if self.curves_output_path:
            self.save_curves()


print('ParallelCurvesParamapAnalysis defined')

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [ ]:
from src.image_loading.options import get_scan_loaders

print('Available scan loaders:', list(get_scan_loaders().keys()))

In [ ]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/CEUS-26152-1.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [ ]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [ ]:
from src.seg_loading.options import get_seg_loaders

print('Available segmentation loaders:', list(get_seg_loaders().keys()))

In [ ]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_vois/v1.1_necrotic_removed.nii.gz'
seg_loader_kwargs = {}

In [ ]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode — Parallel)

In [ ]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

_, all_analysis_funcs = get_analysis_types()
print('Available analysis functions:', list(all_analysis_funcs.keys()))

In [ ]:
import copy

analysis_funcs = ['tic']

# Set frame rate
image_data.frame_rate = 1

analysis_kwargs = {
    'ax_vox_ovrlp': 5.0,
    'sag_vox_ovrlp': 5.0,
    'cor_vox_ovrlp': 5.0,
    'ax_vox_len': 5.0,
    'sag_vox_len': 5.0,
    'cor_vox_len': 5.0,
}

In [ ]:
analyzed_image_data = copy.deepcopy(image_data)

analysis_obj = ParallelCurvesParamapAnalysis(analyzed_image_data, seg_data, analysis_funcs, **analysis_kwargs)
analysis_obj.compute_curves(n_jobs=-1)

print('Analysis object type:', type(analysis_obj))
print('Number of windows:', len(analysis_obj.windows))

## Curve Quantification

In [ ]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print('Available quantification functions:', quantification_funcs.keys())

In [ ]:
function_names = ['lognormal_fit_full']
output_path = '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap/output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC'],
    'tic_name': 'TIC'
}

In [ ]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

print('curve_quant.analysis_objs type:', type(curve_quant.analysis_objs))

## Parametric Map Saving

In [ ]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/tul/china data/p34/new_v1/manual_paramap',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)